In [1]:
import numpy as np

import data.breathe_data as bd
import data.helpers as dh
import models.builders as mb
from plotly.subplots import make_subplots
from plotly import graph_objects as go
import inference.helpers as ih

import pandas as pd
from itertools import chain

In [2]:
df = bd.load_meas_from_excel("BR_O2_FEV1_FEF2575_conservative_smoothing_with_idx")

INFO:root:* Checking for same day measurements *


In [3]:
df.describe()

,FEV1,O2 Saturation,FEF2575,ecFEV1,ecFEF2575,Height,Age,Predicted FEV1,Healthy O2 Saturation,ecFEV1 % Predicted,FEV1 % Predicted,O2 Saturation % Healthy,ecFEF2575%ecFEV1,idx ecFEV1 (L),idx O2 saturation (%),idx ecFEF2575%ecFEV1,idx ecFEF25-75 % ecFEV1 (%)
count,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000,41260.000000
mean,2.270164,97.024891,1.839114,2.281751,1.889478,166.979283,37.261949,3.497435,97.677030,65.144040,64.802971,99.334584,75.899458,45.239942,47.024891,37.452254,37.452254
std,0.872769,1.706424,1.174128,0.868303,1.173371,9.207669,11.469176,0.666026,0.534547,20.428757,20.575953,1.787535,24.898432,17.367107,1.706424,12.383669,12.383669
min,0.300000,75.000000,0.280000,0.400000,0.330000,143.000000,18.000000,2.157715,96.755491,14.155090,10.616317,76.301768,16.064257,8.000000,25.000000,8.000000,8.000000
25%,1.650000,96.000000,0.980000,1.660000,1.030000,160.000000,28.000000,2.938775,97.150104,48.752636,48.264737,98.701631,57.048630,33.000000,46.000000,28.000000,28.000000
50%,2.090000,98.000000,1.460000,2.110000,1.530000,166.000000,36.000000,3.387870,97.971056,63.586063,63.162632,99.792028,70.947833,42.000000,48.000000,35.000000,35.000000
75%,2.800000,98.000000,2.490000,2.810000,2.550000,173.000000,46.000000,4.088245,98.186300,78.405846,78.168586,100.755106,94.296772,56.000000,48.000000,47.000000,47.000000
max,5.980000,100.000000,8.680000,5.980000,8.680000,195.000000,67.000000,5.322753,98.509166,149.505350,148.775632,103.181154,449.740933,119.000000,50.000000,99.000000,99.000000


In [4]:
s_max_ecfev1 = (
    df.groupby("ID").apply(lambda dftmp: dftmp.ecFEV1.max()).rename("max ecFEV1")
)

In [5]:
df_long_fev = bd.load_meas_from_excel(
    "long_model/laplace1.6_30days_fev1",
    str_cols_to_arrays=["AR", "HFEV1"],
).rename(columns={"AR": "AR (fev1)", "HFEV1": "HFEV1 (fev1)"})
df_long_fev_fef = bd.load_meas_from_excel(
    "long_model/laplace1.6_30days_fev1_fef2575",
    str_cols_to_arrays=["AR", "HFEV1"],
).rename(columns={"AR": "AR (fev1 fef2575)", "HFEV1": "HFEV1 (fev1 fef2575)"})

INFO:root:* Checking for same day measurements *
INFO:root:* Checking for same day measurements *


In [6]:
df_fev1 = bd.load_meas_from_excel(
    "infer_AR_using_fev1_01052025", ["AR", "HFEV1", "HO2Sat"], bypass_sanity_checks=True
).drop(columns=["HO2Sat", "HFEV1"])
df_fev1.rename(
    {
        "AR": "AR (fev1 1d)",
        "HFEV1": "HFEV1 (fev1 1d)",
    },
    axis=1,
    inplace=True,
)

In [7]:
df_fev1_fef2575_2d = bd.load_meas_from_excel(
    "infer_AR_using_two_days_model_fev1_fef2575_06052025",
    ["Airway resistance (%)", "Healthy FEV1 (L)"],
    date_cols=["Day"],
    bypass_sanity_checks=True,
)
df_fev1_fef2575_2d.rename(
    {
        "Airway resistance (%)": "AR (fev1 fef2575 2d)",
        "Healthy FEV1 (L)": "HFEV1 (fev1 fef2575 2d)",
        "Day": "Date Recorded",
    },
    axis=1,
    inplace=True,
)

In [8]:
df.columns
cols_2_keep = [
    "ID",
    "Date Recorded",
    "ecFEV1",
    "ecFEV1 % Predicted",
    "ecFEF2575%ecFEV1",
    "Age",
    "Sex",
    "Height",
]
df_agg = df[cols_2_keep].merge(df_long_fev, on=["ID", "Date Recorded"], how="inner")
df_agg = df_agg.merge(s_max_ecfev1, on=["ID"], how="inner")
df_agg = df_agg.merge(
    df_long_fev_fef, on=["ID", "Date Recorded", "Sequence length"], how="inner"
)
df_agg = df_agg.merge(df_fev1_fef2575_2d, on=["ID", "Date Recorded"], how="inner")
df_agg = df_agg.merge(df_fev1, on=["ID", "Date Recorded"], how="inner")
df_agg.head(1)

,ID,Date Recorded,ecFEV1,ecFEV1 % Predicted,ecFEF2575%ecFEV1,Age,Sex,Height,AR (fev1),HFEV1 (fev1),Sequence length,max ecFEV1,AR (fev1 fef2575),HFEV1 (fev1 fef2575),HFEV1 (fev1 fef2575 2d),AR (fev1 fef2575 2d),AR (fev1 1d)
0,101,2019-01-25,1.31,36.287474,41.221374,53,Male,173.0,"[0.00495814436, 0.00955301487, 0.011312835, 0....","[0.0, 2.82791377e-225, 5.36922104e-141, 2.1153...",30,1.79,"[2.99710909e-36, 2.51942291e-34, 1.55251435e-3...","[0.0, 4.54831556e-270, 6.34676627e-186, 2.0096...","[3.51373896e-94, 9.10386601e-81, 8.47365351e-6...","[1.72576618e-19, 2.17342068e-17, 5.81599867e-1...","[6.35674101e-06, 7.86135817e-06, 9.90296787e-0..."


### Viz to compare all models

In [9]:
def plot_for_ID(df_for_ID):
    df_for_ID.reset_index(drop=True, inplace=True)
    if df_for_ID.shape[0] < 20:
        return
    id, age, sex, height, max_ecfev1 = df_for_ID.iloc[0][
        ["ID", "Age", "Sex", "Height", "max ecFEV1"]
    ]
    (
        _,
        _,
        HFEV1,
        _,
        _,
        AR,
        _,
        _,
    ) = mb.fev1_fef2575_long_model_noise_shared_healthy_vars_and_temporal_ar(
        height,
        age,
        sex,
        ar_change_cpt_suffix="_shape_factor_single_laplace_1.6",
        ecfev1_noise_model_suffix="_std_add_mult_ecfev1",
        fef2575_cpt_suffix="",
        light=False,
    )

    def calc_ar_errors(df, ar_col="AR"):
        # mask = df["AR (fev1 fef2575)"].notna()
        df[f"{ar_col}_median"] = df[ar_col].apply(
            lambda x: AR.get_val_at_quantile(x, 0.5)
        )
        df[f"{ar_col}_err_low"] = df[ar_col].apply(
            lambda x: AR.get_val_at_quantile(x, 0.16) - AR.get_val_at_quantile(x, 0.5)
        )
        df[f"{ar_col}_err_high"] = df[ar_col].apply(
            lambda x: AR.get_val_at_quantile(x, 0.5) - AR.get_val_at_quantile(x, 0.84)
        )
        return df

    df_for_ID = calc_ar_errors(df_for_ID, ar_col="AR (fev1 fef2575)")
    df_for_ID = calc_ar_errors(df_for_ID, ar_col="AR (fev1)")
    df_for_ID = calc_ar_errors(df_for_ID, ar_col="AR (fev1 1d)")
    df_for_ID = calc_ar_errors(df_for_ID, ar_col="AR (fev1 fef2575 2d)")

    scatter = {"type": "scatter", "rowspan": 1, "colspan": 1}
    hist = {"type": "histogram", "rowspan": 1, "colspan": 1}
    ar_post = {"type": "bar", "rowspan": 3, "colspan": 1}

    viz_layout = [
        [scatter, ar_post, hist],
        [scatter, None, hist],
        [None, None, None],
    ]

    fig = make_subplots(
        rows=np.shape(viz_layout)[0],
        cols=np.shape(viz_layout)[1],
        specs=viz_layout,
        vertical_spacing=0.08,
        horizontal_spacing=0.07,
        shared_xaxes=True,
        column_widths=[0.3, 0.5, 0.2],
    )

    col = "(fev1 fef2575)"

    # Add ecFEV1
    fig.add_trace(
        go.Scatter(
            y=df_for_ID["ecFEV1"],
            x=df_for_ID["Date Recorded"],
            mode="lines+markers",
        ),
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title="ecFEV1 (L)",
        row=1,
        col=1,
        range=[0, np.nanmax(df.ecFEV1) * 1.02],
        # range=[np.nanmin(df.ecFEV1) * 0.98, np.nanmax(df.ecFEV1) * 1.02],
        title_standoff=19,
    )

    # Add ecFEF2575%ecFEV1
    fig.add_trace(
        go.Scatter(
            y=df_for_ID["ecFEF2575%ecFEV1"],
            x=df_for_ID["Date Recorded"],
            mode="lines+markers",
        ),
        row=2,
        col=1,
    )
    fef2575_min = np.nanmin(df_for_ID["ecFEF2575%ecFEV1"]) * 0.98
    fef2575_max = np.nanmax(df_for_ID["ecFEF2575%ecFEV1"]) * 0.98
    fef2575_mid = fef2575_max - (fef2575_max - fef2575_min) / 2
    fef2575_up = fef2575_mid + 45
    fef2575_low = fef2575_mid - 45
    fef2575_min = int(np.floor(min(fef2575_min, fef2575_low)))
    fef2575_max = int(np.floor(max(fef2575_max, fef2575_up)))
    # print(f"fef2575=[{fef2575_min}, {fef2575_max}], conservative_fef2575=[{fef2575_low}, {fef2575_up}]")
    fig.update_yaxes(
        title="ecFEF25-75%ecFEV1",
        # title="ecFEF2575<br>% ecFEV1",
        row=2,
        col=1,
        range=[
            fef2575_min,
            # 30,
            fef2575_max,
            # 120,
        ],
        title_standoff=5,
    )
    # print(f"range={fef2575_min}, {fef2575_max}")
    fig.data[-2].marker.color = "#0072b2"
    fig.data[-1].marker.color = "#d55e00"

    # ADD AIRWAY RESISTANCE
    def add_ar_trace(fig, df_for_ID, ar_col, row, col):
        fig.add_trace(
            go.Scatter(
                x=df_for_ID["Date Recorded"],
                y=df_for_ID[f"AR {ar_col}_median"],
                mode="markers+lines",
                error_y=dict(
                    type="data",
                    symmetric=False,
                    array=df_for_ID[f"AR {ar_col}_err_high"],
                    arrayminus=df_for_ID[f"AR {ar_col}_err_low"],
                    thickness=0.5,
                    width=4,
                    color="black",
                ),
                name=f"AR {ar_col}",
                marker=dict(color="#0072b2" if ar_col == "(fev1)" else "#d55e00"),
                # marker=dict(size=8, color="steelblue", line=dict(width=0.5, color="black")),
            ),
            row=row,
            col=col,
        )

    # ADD AIRWAY RESISTANCE against 2 days model, against ecFEV1%
    fig.add_trace(
        go.Scatter(
            x=df_for_ID["Date Recorded"],
            y=(100 - df_for_ID["ecFEV1 % Predicted"]).apply(lambda x: max(x, 0)),
            mode="markers+lines",
            name="1-ecFEV1%",
        ),
        row=1,
        col=2,
    )
    add_ar_trace(fig, df_for_ID, "(fev1 1d)", 1, 2)
    add_ar_trace(fig, df_for_ID, "(fev1 fef2575 2d)", 1, 2)
    add_ar_trace(fig, df_for_ID, "(fev1)", 1, 2)
    add_ar_trace(fig, df_for_ID, "(fev1 fef2575)", 1, 2)
    fig.data[-5].marker.color = "black"
    fig.data[-4].marker.color = "#f0e442"
    fig.data[-3].marker.color = "#009e73"
    fig.data[-2].marker.color = "#0072b2"
    fig.data[-1].marker.color = "#d55e00"
    range_min = np.min(
        list(
            chain.from_iterable(
                [
                    (100 - df_for_ID["ecFEV1 % Predicted"]).apply(lambda x: max(x, 0)),
                    df_for_ID["AR (fev1)_median"] + df_for_ID[f"AR (fev1)_err_low"],
                    df_for_ID["AR (fev1 1d)_median"]
                    + df_for_ID[f"AR (fev1 1d)_err_low"],
                    df_for_ID["AR (fev1 fef2575)_median"]
                    + df_for_ID[f"AR (fev1 fef2575)_err_low"],
                    df_for_ID["AR (fev1 fef2575 2d)_median"]
                    + df_for_ID[f"AR (fev1 fef2575 2d)_err_low"],
                ]
            )
        )
    )
    range_max = np.max(
        list(
            chain.from_iterable(
                [
                    (100 - df_for_ID["ecFEV1 % Predicted"]).apply(lambda x: max(x, 0)),
                    df_for_ID["AR (fev1)_median"] - df_for_ID[f"AR (fev1)_err_high"],
                    df_for_ID["AR (fev1 1d)_median"]
                    - df_for_ID[f"AR (fev1 1d)_err_high"],
                    df_for_ID["AR (fev1 fef2575)_median"]
                    - df_for_ID[f"AR (fev1 fef2575)_err_high"],
                    df_for_ID["AR (fev1 fef2575 2d)_median"]
                    - df_for_ID[f"AR (fev1 fef2575 2d)_err_high"],
                ]
            )
        )
    )
    range_mid = range_max - np.round((range_max - range_min) / 2)
    range_up = range_mid + 25
    range_low = range_mid - 25
    # print(f"range=[{range_min}, {range_max}], conservative_range=[{range_low}, {range_up}]")
    range_min = int(np.floor(min(range_min, range_low)))
    range_max = int(np.ceil(max(range_max, range_up)))
    # print(range_min, range_max)
    fig.update_yaxes(
        row=1,
        col=2,
        range=[range_min, range_max],
        title_text=AR.name,
        tickmode="array",
        tickvals=np.linspace(AR.a, AR.b, 10),
        title_standoff=10,
    )

    # ADD HFEV1
    ih.plot_histogram(
        fig, HFEV1, HFEV1.cpt, 0, HFEV1.b, 1, 3, annot=False, colour="lightblue"
    )
    ih.plot_histogram(
        fig, HFEV1, HFEV1.cpt, 0, HFEV1.b, 2, 3, annot=False, colour="lightblue"
    )

    ar_cols = ["(fev1)", "(fev1 fef2575)"]
    ih.plot_histogram(
        fig,
        HFEV1,
        df_for_ID[f"HFEV1 {ar_cols[0]}"][0],
        0,
        HFEV1.b,
        1,
        3,
        annot=False,
        opacity=1,
    )
    ih.plot_histogram(
        fig,
        HFEV1,
        df_for_ID[f"HFEV1 {ar_cols[1]}"][0],
        0,
        HFEV1.b,
        2,
        3,
        annot=False,
        opacity=1,
    )

    # Change last  traces colours
    fig.data[-2].marker.color = "#0072b2"
    fig.data[-1].marker.color = "#d55e00"

    p_max = max(
        df_for_ID[f"HFEV1 {ar_cols[0]}"][0].max(),
        df_for_ID[f"HFEV1 {ar_cols[1]}"][0].max(),
    )
    fig.update_yaxes(
        title_text="Probability",
        range=[0, p_max * 1.1],
        title_standoff=5,
        row=1,
        col=3,
        showticklabels=False,
        showgrid=False,
    )
    fig.update_yaxes(
        title_text="Probability",
        range=[0, p_max * 1.1],
        row=2,
        col=3,
        title_standoff=5,
        showticklabels=False,
        showgrid=False,
    )
    fig.update_xaxes(
        row=2,
        col=3,
        tickmode="array",
        tickvals=np.arange(7),
        ticktext=[str(i) for i in range(7)],
        title_text=HFEV1.name,
        title_standoff=5,
    )

    fig.update_xaxes(
        # showticklabels=False,
        showgrid=False,
    )
    # fig.update_yaxes(
    #     # showticklabels=False,
    #     showgrid=False,
    # )

    title = f"{id} - Longitudinal model results - {len(df_for_ID)} entries, max ecFEV1={max_ecfev1}L"
    fig.update_layout(
        font=dict(size=11),
        height=550,
        width=1200,
        title=title,
        title_font_size=14,
        showlegend=True,
        barmode="overlay",
    )

    fig.show()
    # fig.write_image(
    #     f"{dh.get_path_to_main()}/PlotsBreathe/Long model short term/{title}.pdf"
    # )


plot_for_ID(df_agg[df_agg.ID == "101"][0:20])

In [29]:
df_agg.groupby("ID").apply(plot_for_ID)

101
102
103
106
107
109
111
112
116
117
120
122
123
125
126
127
130
133
138
139
140
145
146
147
148
151
153
159
162
163
165
170
172
180
182
184
185
188
196
198
201
203
215
221
229
230
237
238
240
244
250
254
272
282
311
331
334
336
339
352
361
365
381
405
411
420
426
452
469
480
483
493
502
506
508
509
515
517
520
523
530
537
544


""


### Viz to showcase model capacity

In [80]:
def plot_for_ID(df_for_ID):
    df_for_ID.reset_index(drop=True, inplace=True)
    if df_for_ID.shape[0] < 20:
        return
    id, age, sex, height, max_ecfev1 = df_for_ID.iloc[0][
        ["ID", "Age", "Sex", "Height", "max ecFEV1"]
    ]
    (
        _,
        _,
        HFEV1,
        _,
        _,
        AR,
        _,
        _,
    ) = mb.fev1_fef2575_long_model_noise_shared_healthy_vars_and_temporal_ar(
        height,
        age,
        sex,
        ar_change_cpt_suffix="_shape_factor_single_laplace_1.6",
        ecfev1_noise_model_suffix="_std_add_mult_ecfev1",
        fef2575_cpt_suffix="",
        light=False,
    )

    def calc_one_minus_ar_errors(df, ar_col="AR"):
        # mask = df["AR (fev1 fef2575)"].notna()
        df[f"1-{ar_col}_median"] = 100 - df[ar_col].apply(
            lambda x: AR.get_val_at_quantile(x, 0.5)
        )
        df[f"1-{ar_col}_err_high"] = df[ar_col].apply(
            lambda x: AR.get_val_at_quantile(x, 0.16) - AR.get_val_at_quantile(x, 0.5)
        )
        df[f"1-{ar_col}_err_low"] = df[ar_col].apply(
            lambda x: AR.get_val_at_quantile(x, 0.5) - AR.get_val_at_quantile(x, 0.84)
        )
        return df

    df_for_ID = calc_one_minus_ar_errors(df_for_ID, ar_col="AR (fev1 fef2575)")

    scatter = {"type": "scatter", "rowspan": 1, "colspan": 1}
    hist = {"type": "histogram", "rowspan": 1, "colspan": 1}
    ar_post = {"type": "bar", "rowspan": 3, "colspan": 1}

    viz_layout = [
        [scatter, ar_post, hist],
        [scatter, None, hist],
        [None, None, None],
    ]

    fig = make_subplots(
        rows=np.shape(viz_layout)[0],
        cols=np.shape(viz_layout)[1],
        specs=viz_layout,
        vertical_spacing=0.08,
        horizontal_spacing=0.07,
        shared_xaxes=True,
        column_widths=[0.3, 0.4, 0.3],
    )

    col = "(fev1 fef2575)"

    # Add ecFEV1
    fig.add_trace(
        go.Scatter(
            y=df_for_ID["ecFEV1"],
            x=df_for_ID["Date Recorded"],
            mode="lines+markers",
        ),
        row=1,
        col=1,
    )
    fig.update_yaxes(
        title="ecFEV1 (L)",
        row=1,
        col=1,
        range=[0, np.nanmax(df.ecFEV1) * 1.02],
        # range=[np.nanmin(df.ecFEV1) * 0.98, np.nanmax(df.ecFEV1) * 1.02],
        title_standoff=19,
    )

    # Add ecFEF2575%ecFEV1
    fig.add_trace(
        go.Scatter(
            y=df_for_ID["ecFEF2575%ecFEV1"],
            x=df_for_ID["Date Recorded"],
            mode="lines+markers",
        ),
        row=2,
        col=1,
    )
    fef2575_min = np.nanmin(df_for_ID["ecFEF2575%ecFEV1"]) * 0.98
    fef2575_max = np.nanmax(df_for_ID["ecFEF2575%ecFEV1"]) * 0.98
    fef2575_mid = fef2575_max - (fef2575_max - fef2575_min) / 2
    fef2575_up = fef2575_mid + 45
    fef2575_low = fef2575_mid - 45
    fef2575_min = int(np.floor(min(fef2575_min, fef2575_low)))
    fef2575_max = int(np.floor(max(fef2575_max, fef2575_up)))
    # print(f"fef2575=[{fef2575_min}, {fef2575_max}], conservative_fef2575=[{fef2575_low}, {fef2575_up}]")
    fig.update_yaxes(
        title="ecFEF25-75%ecFEV1",
        # title="ecFEF2575<br>% ecFEV1",
        row=2,
        col=1,
        range=[
            fef2575_min,
            # 30,
            fef2575_max,
            # 120,
        ],
        title_standoff=5,
    )
    # print(f"range={fef2575_min}, {fef2575_max}")
    fig.data[-2].marker.color = "#0072b2"
    fig.data[-1].marker.color = "#d55e00"

    # ADD AIRWAY RESISTANCE
    def add_ar_trace(fig, df_for_ID, ar_col, row, col):
        fig.add_trace(
            go.Scatter(
                x=df_for_ID["Date Recorded"],
                y=df_for_ID[f"1-AR {ar_col}_median"],
                mode="markers+lines",
                error_y=dict(
                    type="data",
                    symmetric=False,
                    array=df_for_ID[f"1-AR {ar_col}_err_high"],
                    arrayminus=df_for_ID[f"1-AR {ar_col}_err_low"],
                    thickness=1,
                    width=5,
                    # color="black",
                ),
                # name=f"1-AR {ar_col} (model result)",
                name=f"f(FEV1 % predicted) (model result)",
                marker=dict(color="#0072b2" if ar_col == "(fev1)" else "#d55e00"),
                # marker=dict(size=8, color="steelblue", line=dict(width=0.5, color="black")),
            ),
            row=row,
            col=col,
        )

    # ADD AIRWAY RESISTANCE against 2 days model, against ecFEV1%
    fig.add_trace(
        go.Scatter(
            x=df_for_ID["Date Recorded"],
            y=df_for_ID["ecFEV1 % Predicted"].apply(lambda x: max(x, 0)),
            mode="markers+lines",
            name="ecFEV1 % predicted (clinical standard)",
        ),
        row=1,
        col=2,
    )
    add_ar_trace(fig, df_for_ID, "(fev1 fef2575)", 1, 2)
    fig.data[-2].marker.color = "black"
    fig.data[-1].marker.color = "#d55e00"
    range_min = np.min(
        list(
            chain.from_iterable(
                [
                    (df_for_ID["ecFEV1 % Predicted"]).apply(lambda x: x),
                    df_for_ID["1-AR (fev1 fef2575)_median"]
                    + df_for_ID[f"1-AR (fev1 fef2575)_err_low"],
                ]
            )
        )
    )
    range_max = np.max(
        list(
            chain.from_iterable(
                [
                    df_for_ID["ecFEV1 % Predicted"].apply(lambda x: min(x, 100)),
                    df_for_ID["1-AR (fev1 fef2575)_median"]
                    - df_for_ID[f"1-AR (fev1 fef2575)_err_high"],
                ]
            )
        )
    )
    # print(f"ranges {range_min} → {range_max}")
    range_mid = range_max - np.round((range_max - range_min) / 2)
    range_up = range_mid + 25
    range_low = range_mid - 25
    # print(f"range=[{range_min}, {range_max}], conservative_range=[{range_low}, {range_up}]")
    range_min = int(np.floor(min(range_min, range_low)))
    range_max = int(np.ceil(max(range_max, range_up)))
    # print(range_min, range_max)
    fig.update_yaxes(
        row=1,
        col=2,
        range=[range_min, range_max],
        title_text="Lung health (%)",
        tickmode="array",
        tickvals=np.linspace(0, 120, 13),
        title_standoff=10,
    )

    # ADD HFEV1
    ih.plot_histogram(
        fig, HFEV1, HFEV1.cpt, 0, HFEV1.b, 1, 3, annot=False, colour="lightblue"
    )
    ih.plot_histogram(
        fig, HFEV1, HFEV1.cpt, 0, HFEV1.b, 2, 3, annot=False, colour="lightblue"
    )

    ar_cols = ["(fev1)", "(fev1 fef2575)"]
    ih.plot_histogram(
        fig,
        HFEV1,
        df_for_ID[f"HFEV1 {ar_cols[0]}"][0],
        0,
        HFEV1.b,
        1,
        3,
        annot=False,
        opacity=1,
    )
    ih.plot_histogram(
        fig,
        HFEV1,
        df_for_ID[f"HFEV1 {ar_cols[1]}"][0],
        0,
        HFEV1.b,
        2,
        3,
        annot=False,
        opacity=1,
    )

    # Change last  traces colours
    fig.data[-2].marker.color = "#0072b2"
    fig.data[-1].marker.color = "#d55e00"

    p_max = max(
        df_for_ID[f"HFEV1 {ar_cols[0]}"][0].max(),
        df_for_ID[f"HFEV1 {ar_cols[1]}"][0].max(),
    )
    fig.update_yaxes(
        title_text="Probability",
        range=[0, p_max * 1.1],
        title_standoff=5,
        row=1,
        col=3,
        showticklabels=False,
        showgrid=False,
    )
    fig.update_yaxes(
        title_text="Probability",
        range=[0, p_max * 1.1],
        row=2,
        col=3,
        title_standoff=5,
        showticklabels=False,
        showgrid=False,
    )
    fig.update_xaxes(
        row=2,
        col=3,
        tickmode="array",
        tickvals=np.arange(7),
        ticktext=[str(i) for i in range(7)],
        title_text=HFEV1.name,
        title_standoff=5,
    )

    fig.update_xaxes(
        # showticklabels=False,
        showgrid=False,
    )
    # fig.update_yaxes(
    #     # showticklabels=False,
    #     showgrid=False,
    # )

    title = f"{id} - Longitudinal model results - {len(df_for_ID)} entries, max ecFEV1={max_ecfev1}L"
    fig.update_layout(
        font=dict(size=11),
        height=550,
        width=1300,
        title=title,
        title_font_size=14,
        showlegend=True,
        barmode="overlay",
    )

    # fig.show()
    fig.write_image(
        f"{dh.get_path_to_main()}/PlotsBreathe/Long model short term/1-AR {title}.pdf"
    )


# plot_for_ID(df_agg[df_agg.ID == "101"][0:20])

In [81]:
df_agg.groupby("ID").apply(lambda df_for_ID: plot_for_ID(df_for_ID))

""
